# Q07 예비 확인 (B 방식: 인덱스 없음)

- 공동 DB에 **인덱스를 만들지 않는다** (팀 승인 전, 목록 항목 1).
- 블록 전체를 테스트 라벨(`PLTest`)과 접두사(`test_`)로 넣고, 키워드 점수로 순위를 본다.
- **확인 목적**: Q07에 필요한 근거(학교·의료기관·법 조건·보고)가 상위 후보에 같이 나오는지 본다.
- **한계**: 전체 검색 성능이 아니다. 예비 결과다. 공식 게이트 통과가 아니다.

In [1]:
import json
from rag.prevention_legal.neo4j_conn import use_windows_trust, connect

use_windows_trust()
driver = connect()
TEST_LABEL, PREFIX = "PLTest", "test_"

with driver.session() as s:
    before = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
print("시작 전 전체 노드:", before)

시작 전 전체 노드: 275


In [2]:
blocks = [json.loads(l) for l in open("../../data/interim/prevention_legal/blocks.jsonl", encoding="utf-8")]
CYPHER = f"""
MERGE (c:{TEST_LABEL} {{id: $id}})
SET c.document_id = $document_id, c.pdf_page = $pdf_page,
    c.section = $section, c.text = $text, c.is_test = true
"""
with driver.session() as s:
    for b in blocks:
        s.run(CYPHER, id=PREFIX + b["chunk_id"], document_id=b["document_id"],
              pdf_page=b["pdf_page"], section=b.get("section") or "", text=b["text"])
print("테스트 블록 적재:", len(blocks))

테스트 블록 적재: 25


In [3]:
# Q07 질문 검색어 (질문의 핵심어). 학교·의료기관·법 조건·보고를 모두 겨냥한다.
Q07 = "홍역 의심환자가 생겼을 때 학교 담당교사와 의료기관은 각각 보건소에 무엇을 신고하거나 안내하나요? 법률과 지침의 조건을 구분해 주세요."
WORDS = ["홍역", "의심환자", "학교", "담당교사", "의료기관", "보건소", "신고", "보고", "진료", "안내", "관리인"]

# 기대 근거 (Q07 답에 필요한 블록)
EXPECTED = {
    "kdca_measles_2026:p0093:b01": "학교 담당교사: 진료 안내·보건소 신고",
    "moleg_infectious_disease_act_21521:p0018:b01": "의사: 소속 의료기관장 보고",
    "moleg_infectious_disease_act_21521:p0018:b02": "의료기관장: 1·2·3·4급 신고 기한",
    "moleg_infectious_disease_act_21521:p0019:b01": "학교 관리인 등: 보건복지부령 감염병 신고",
    "moleg_infectious_disease_act_21521:p0019:b03": "보건소장: 상급 기관 보고",
}

with driver.session() as s:
    rows = s.run(f"MATCH (c:{TEST_LABEL}) WHERE c.id STARTS WITH $p RETURN c.id AS id, c.text AS text, c.section AS section",
                 p=PREFIX).data()

def score(r):
    hay = (r["text"] or "") + " " + (r["section"] or "")
    return sum(hay.count(w) for w in WORDS)

ranked = sorted(rows, key=lambda r: score(r), reverse=True)
print("상위 10개 (점수 순):")
for i, r in enumerate(ranked[:10], 1):
    print(f"  {i:2d}위 | 점수 {score(r):2d} | {r['id'][len(PREFIX):]}")

print()
print("기대 근거의 순위:")
pos = {r["id"][len(PREFIX):]: i for i, r in enumerate(ranked, 1)}
for cid, desc in EXPECTED.items():
    rank = pos.get(cid)
    mark = "상위 5위 안 ✅" if rank and rank <= 5 else ("상위 10위 안" if rank and rank <= 10 else "10위 밖 ❌")
    print(f"  {rank:2d}위 | {mark} | {desc} | {cid}")

상위 10개 (점수 순):
   1위 | 점수  9 | kdca_measles_2026:p0093:b01
   2위 | 점수  9 | moleg_infectious_disease_act_21521:p0018:b01
   3위 | 점수  9 | kdca_diagnosis_reporting_2026:p0089:b04
   4위 | 점수  9 | moleg_infectious_disease_act_21521:p0019:b03
   5위 | 점수  8 | moleg_infectious_disease_act_21521:p0019:b01
   6위 | 점수  8 | kdca_diagnosis_reporting_2026:p0104:b04
   7위 | 점수  7 | moleg_infectious_disease_act_21521:p0018:b02
   8위 | 점수  6 | kdca_measles_2026:p0093:b03
   9위 | 점수  6 | kdca_measles_2026:p0030:b02
  10위 | 점수  5 | moleg_infectious_disease_act_21521:p0019:b02

기대 근거의 순위:
   1위 | 상위 5위 안 ✅ | 학교 담당교사: 진료 안내·보건소 신고 | kdca_measles_2026:p0093:b01
   2위 | 상위 5위 안 ✅ | 의사: 소속 의료기관장 보고 | moleg_infectious_disease_act_21521:p0018:b01
   7위 | 상위 10위 안 | 의료기관장: 1·2·3·4급 신고 기한 | moleg_infectious_disease_act_21521:p0018:b02
   5위 | 상위 5위 안 ✅ | 학교 관리인 등: 보건복지부령 감염병 신고 | moleg_infectious_disease_act_21521:p0019:b01
   4위 | 상위 5위 안 ✅ | 보건소장: 상급 기관 보고 | moleg_infectious_disease_act_21521:p0019:b03


In [4]:
# 정리: 테스트 데이터만 삭제하고 0개인지 확인
with driver.session() as s:
    s.run(f"MATCH (c:{TEST_LABEL}) WHERE c.id STARTS WITH $p DETACH DELETE c", p=PREFIX)
    left = s.run(f"MATCH (c:{TEST_LABEL}) RETURN count(c) AS c").single()["c"]
    after = s.run("MATCH (n) RETURN count(n) AS c").single()["c"]
print("삭제 후 테스트 노드:", left, "| 전체 노드:", after)
driver.close()

삭제 후 테스트 노드: 0 | 전체 노드: 275
